In [1]:
from pyspark.sql import functions as F

# ============================================================
# NB_Gold_Fact_Sales
# Silver -> Gold
# Grain: one row per order item
# ============================================================

ORDER_ITEMS_TABLE = "silver_order_items"
ORDERS_TABLE = "silver_orders"

CUSTOMER_DIM = "dim_customer"
PRODUCT_DIM = "dim_product"
DATE_DIM = "dim_date"

TARGET_TABLE = "fact_sales"

# ------------------------------------------------------------
# 1. Read Silver tables
# ------------------------------------------------------------

order_items = spark.table(ORDER_ITEMS_TABLE)
orders = spark.table(ORDERS_TABLE)

# ------------------------------------------------------------
# 2. Read dimensions
# ------------------------------------------------------------

customers = (
    spark.table(CUSTOMER_DIM)
    .select(
        "customer_key",
        "customer_id"
    )
)

products = (
    spark.table(PRODUCT_DIM)
    .select(
        "product_key",
        "product_id"
    )
)

dates = (
    spark.table(DATE_DIM)
    .select(
        "date_key",
        "date"
    )
)

# ------------------------------------------------------------
# 3. Join order items with orders
# ------------------------------------------------------------

sales = (
    order_items
    .join(
        orders.select(
            "order_id",
            "customer_id",
            "order_date",
            "status"
        ),
        on="order_id",
        how="inner"
    )
)

# ------------------------------------------------------------
# 4. Join customer dimension
# ------------------------------------------------------------

sales = (
    sales
    .join(
        customers,
        on="customer_id",
        how="left"
    )
)

# ------------------------------------------------------------
# 5. Join product dimension
# ------------------------------------------------------------

sales = (
    sales
    .join(
        products,
        on="product_id",
        how="left"
    )
)

# ------------------------------------------------------------
# 6. Derive order date
# ------------------------------------------------------------

sales = sales.withColumn(
    "order_date_only",
    F.to_date("order_date")
)

# ------------------------------------------------------------
# 7. Join date dimension
# ------------------------------------------------------------

sales = (
    sales
    .join(
        dates,
        sales.order_date_only == dates.date,
        how="left"
    )
)

# ------------------------------------------------------------
# 8. Calculate sales amount
# ------------------------------------------------------------

sales = sales.withColumn(
    "sales_amount",
    F.round(
        F.col("quantity") * F.col("unit_price"),
        2
    )
)

# ------------------------------------------------------------
# 9. Build Gold fact
# ------------------------------------------------------------

df_gold = (
    sales
    .select(
        "order_item_id",
        "order_id",
        "customer_key",
        "product_key",
        "date_key",
        "quantity",
        "unit_price",
        "sales_amount",
        "status"
    )
    .withColumn(
        "item_count",
        F.lit(1)
    )
)

# ------------------------------------------------------------
# 10. Write Gold fact
# ------------------------------------------------------------

df_gold.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable(TARGET_TABLE)

# ------------------------------------------------------------
# 11. Validation
# ------------------------------------------------------------

print(f"Gold table created: {TARGET_TABLE}")
print(f"Records: {df_gold.count()}")

df_gold.show(10, truncate=False)

StatementMeta(, 8a35d219-00ef-4a1c-9804-b86e5d1cf073, 3, Finished, Available, Finished, False)

Gold table created: fact_sales
Records: 119569
+-------------+--------+------------+-----------+--------+--------+----------+------------+---------+----------+
|order_item_id|order_id|customer_key|product_key|date_key|quantity|unit_price|sales_amount|status   |item_count|
+-------------+--------+------------+-----------+--------+--------+----------+------------+---------+----------+
|2            |42850   |537         |654        |20260421|1       |1053.86   |1053.86     |delivered|1         |
|4            |11043   |7787        |469        |20241109|5       |1671.86   |8359.30     |delivered|1         |
|7            |42888   |5500        |86         |20241101|3       |497.06    |1491.18     |delivered|1         |
|8            |19601   |1483        |851        |20241113|3       |276.36    |829.08      |shipped  |1         |
|9            |18947   |8465        |715        |20251024|2       |1766.07   |3532.14     |delivered|1         |
|10           |31607   |5879        |774        |